# 01 Preprocessing

Preprocessing-only notebook for loading cleaned ASHRAE data and building engineered features.


## Scope
- Load cleaned train/weather/building metadata tables
- Merge and validate columns
- Build preprocessing features (time, weather, metadata, lag/rolling, encodings)


## 1. Environment Setup and Data Loading

In [1]:
import os, random, math, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean, cosine as cosine_dist
from scipy.signal import savgol_filter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1a. Generate Cleaned Input Files
Run these once (or whenever raw data/preprocessing params change) before loading data below.


In [2]:
import subprocess, sys

RUN_PREPROCESS_SCRIPTS = True  

if RUN_PREPROCESS_SCRIPTS:
    cmd = [sys.executable, "ashrae/preprocess_isamu_matt.py", "--input-dir", "ashrae", "--output-dir", "ashrae"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped: preprocess_isamu_matt.py")


Running: c:\Users\tamar\AppData\Local\Programs\Python\Python314\python.exe ashrae/preprocess_isamu_matt.py --input-dir ashrae --output-dir ashrae


In [3]:
import subprocess, sys

RUN_MEAN_FILTER_SCRIPT = True  

if RUN_MEAN_FILTER_SCRIPT:
    cmd = [sys.executable, "ashrae/make_building_mean_y_ge_1.py"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipped: make_building_mean_y_ge_1.py")


Running: c:\Users\tamar\AppData\Local\Programs\Python\Python314\python.exe ashrae/make_building_mean_y_ge_1.py


## Result
df contains the preprocessed dataset ready for downstream training.
